In [ ]:
# Setup: imports, seeds, and a parameterizable CartPole wrapper
import gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import random
from collections import deque
from typing import Callable

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# --- Configurable Environment Wrapper ---
class ParameterizedCartPole(gym.Wrapper):
    def __init__(self, env, gravity=9.8, masscart=1.0):
        super().__init__(env)
        self.gravity = gravity
        self.masscart = masscart

    def reset(self, **kwargs):
        # Inject physics parameters into the Gym env (CartPole implementation exposes these on unwrapped)
        try:
            self.env.unwrapped.gravity = self.gravity
            self.env.unwrapped.masscart = self.masscart
        except Exception:
            # Some Gym versions may not expose these attributes; ignore if not available
            pass
        return self.env.reset(**kwargs)

def make_env(gravity: float = 9.8, mass: float = 1.0):
    env = gym.make("CartPole-v1")
    return ParameterizedCartPole(env, gravity=gravity, masscart=mass)

# Utility: simple plotting helper
def plot_rewards(rewards, label="Rewards", smooth=5):
    sm = np.convolve(rewards, np.ones(smooth) / smooth, mode='valid')
    plt.plot(sm, label=label)
    plt.xlabel('Episode (smoothed)')
    plt.ylabel('Return')
    plt.legend()
    plt.grid(True)

In [ ]:
# --- DQN & Agent definitions ---
class DQN(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, action_dim)
        )
    def forward(self, x):
        return self.net(x)

class Agent:
    def __init__(self, state_dim, action_dim, lr=1e-3, gamma=0.99):
        self.q_net = DQN(state_dim, action_dim)
        self.target_net = DQN(state_dim, action_dim)
        self.target_net.load_state_dict(self.q_net.state_dict())
        self.optimizer = optim.Adam(self.q_net.parameters(), lr=lr)
        self.gamma = gamma
        self.memory = deque(maxlen=10000)
        self.batch_size = 64
        self.action_dim = action_dim
        self.epsilon = 1.0
        self.epsilon_decay = 0.995
        self.epsilon_min = 0.05
        self.loss_fn = nn.MSELoss()

    def act(self, state, train=True):
        if train and np.random.rand() < self.epsilon:
            return np.random.randint(self.action_dim)
        state_t = torch.FloatTensor(state).unsqueeze(0)
        with torch.no_grad():
            q_vals = self.q_net(state_t)
        return int(q_vals.argmax().item())

    def remember(self, s, a, r, ns, d):
        self.memory.append((s, a, r, ns, d))

    def train_step(self):
        if len(self.memory) < self.batch_size:
            return
        batch = random.sample(self.memory, self.batch_size)
        s, a, r, ns, d = zip(*batch)
        s = torch.FloatTensor(s)
        a = torch.LongTensor(a).unsqueeze(1)
        r = torch.FloatTensor(r).unsqueeze(1)
        ns = torch.FloatTensor(ns)
        d = torch.FloatTensor(d).unsqueeze(1)

        with torch.no_grad():
            q_next = self.target_net(ns).max(1, keepdim=True)[0]
            target = r + (1 - d) * self.gamma * q_next

        q_curr = self.q_net(s).gather(1, a)
        loss = self.loss_fn(q_curr, target)

        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        if self.epsilon > self.epsilon_min:
            self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

    def update_target(self):
        self.target_net.load_state_dict(self.q_net.state_dict())

# --- Training Loop Helper ---
def train_agent(env_maker: Callable[[], gym.Env], episodes=200, label="Agent", randomize=False):
    env = env_maker()
    agent = Agent(env.observation_space.shape[0], env.action_space.n)
    rewards = []

    for ep in range(episodes):
        # Domain Randomization (if enabled)
        if randomize:
            g = np.random.uniform(5.0, 15.0)
            env = make_env(gravity=g)

        reset_out = env.reset()
        # gym/gymnasium compatibility: reset may return (obs, info)
        state = reset_out[0] if isinstance(reset_out, tuple) else reset_out
        total_r = 0
        done = False
        while not done:
            action = agent.act(state)
            next_out = env.step(action)
            next_state = next_out[0] if isinstance(next_out[0], np.ndarray) or True else next_out[0]
            reward = next_out[1]
            terminated = next_out[2]
            truncated = next_out[3] if len(next_out) > 3 else False
            done = terminated or truncated

            agent.remember(state, action, reward, next_state, done)
            agent.train_step()
            state = next_state
            total_r += reward

        if ep % 10 == 0:
            agent.update_target()
        rewards.append(total_r)

    return agent, rewards

## Part 1 — The Generalization Gap & Domain Randomization ✅

We'll train two agents:
- **Baseline:** trained on Earth gravity (9.8).
- **Randomized:** trained with domain randomization (gravity drawn from [5, 15]).

We'll evaluate both on a **target** high-gravity domain (e.g., 25.0) to demonstrate transfer.

In [ ]:
# Train Baseline and Randomized agents (short runs for example purposes)
print("Training Baseline on Source (Gravity=9.8)...")
source_agent, src_rewards = train_agent(lambda: make_env(gravity=9.8), episodes=150)

print("Training Randomized Agent (Gravity=5.0 to 15.0)...")
rand_agent, rand_rewards = train_agent(lambda: make_env(gravity=9.8), episodes=150, randomize=True)

# Evaluate function
def evaluate(agent: Agent, gravity: float, episodes: int = 10):
    env = make_env(gravity=gravity)
    scores = []
    for _ in range(episodes):
        reset_out = env.reset()
        s = reset_out[0] if isinstance(reset_out, tuple) else reset_out
        score = 0
        done = False
        while not done:
            a = agent.act(s, train=False)
            next_out = env.step(a)
            s = next_out[0]
            r = next_out[1]
            term = next_out[2]
            trunc = next_out[3] if len(next_out) > 3 else False
            done = term or trunc
            score += r
        scores.append(score)
    return np.mean(scores)

# Evaluate on a strong gravity target (25.0)
score_baseline = evaluate(source_agent, gravity=25.0)
score_random = evaluate(rand_agent, gravity=25.0)

print(f"\n--- Zero-Shot Transfer Results (Target Gravity=25.0) ---")
print(f"Baseline Agent Score:   {score_baseline:.2f} (Likely Failed)")
print(f"Randomized Agent Score: {score_random:.2f} (Better Generalization)")

plt.figure(figsize=(8,4))
plt.bar(["Baseline (Source Only)", "Domain Randomized"], [score_baseline, score_random], color=['red', 'green'])
plt.ylabel("Average Return on Target Domain")
plt.title("Impact of Domain Randomization on Transfer")
plt.show()

# Plot training curves (smoothed)
plt.figure(figsize=(10,4))
plot_rewards(src_rewards, label='Baseline')
plot_rewards(rand_rewards, label='Domain Randomized')
plt.title('Training Returns (smoothed)')
plt.show()

## Part 2 — Dynamics Adaptation (Sim-to-Real) 🔧

We'll collect random transitions from both the simulator (source) and a high-gravity target (serving as "real"), train a discriminator to distinguish them, then use the discriminator's log-odds as a reward offset while re-training on the simulator.

In [ ]:
# --- Discriminator & Data Collection ---
class DynamicsDiscriminator(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        # Input: state + action + next_state
        self.net = nn.Sequential(
            nn.Linear(state_dim * 2 + 1, 64), nn.ReLU(),
            nn.Linear(64, 1), nn.Sigmoid()
        )
    def forward(self, s, a, ns):
        cat = torch.cat([s, a, ns], dim=1)
        return self.net(cat)


def collect_transitions(env, episodes=20):
    data = []
    reset_out = env.reset()
    state = reset_out[0] if isinstance(reset_out, tuple) else reset_out
    for _ in range(episodes):
        done = False
        while not done:
            action = env.action_space.sample()  # Random policy for data collection
            next_out = env.step(action)
            next_state = next_out[0]
            terminated = next_out[2]
            truncated = next_out[3] if len(next_out) > 3 else False
            done = terminated or truncated
            data.append((state, action, next_state))
            state = next_state
            if done:
                reset_out = env.reset()
                state = reset_out[0] if isinstance(reset_out, tuple) else reset_out
    return data

print("Collecting Dynamics Data...")
source_data = collect_transitions(make_env(gravity=9.8), episodes=50)   # Simulator
target_data = collect_transitions(make_env(gravity=25.0), episodes=50)  # "Real World"

# Sanity
print(f"Collected {len(source_data)} source and {len(target_data)} target transitions.")

# Train Discriminator
discriminator = DynamicsDiscriminator(state_dim=4, action_dim=1)
opt_d = optim.Adam(discriminator.parameters(), lr=1e-3)
loss_fn = nn.BCELoss()

print("Training Dynamics Discriminator...")
for epoch in range(500):
    # Sample batches
    src_batch = random.sample(source_data, 32)
    tgt_batch = random.sample(target_data, 32)

    def prepare(batch):
        s, a, ns = zip(*batch)
        s = torch.FloatTensor(s)
        a = torch.FloatTensor(a).unsqueeze(1)
        ns = torch.FloatTensor(ns)
        return s, a, ns

    s_s, a_s, ns_s = prepare(src_batch)
    s_t, a_t, ns_t = prepare(tgt_batch)

    pred_src = discriminator(s_s, a_s, ns_s)
    pred_tgt = discriminator(s_t, a_t, ns_t)

    loss = loss_fn(pred_src, torch.zeros_like(pred_src)) + loss_fn(pred_tgt, torch.ones_like(pred_tgt))

    opt_d.zero_grad()
    loss.backward()
    opt_d.step()

    if epoch % 100 == 0:
        with torch.no_grad():
            acc_src = ((pred_src < 0.5).float().mean().item())
            acc_tgt = ((pred_tgt > 0.5).float().mean().item())
        print(f"Epoch {epoch:03d}  loss={loss.item():.4f}  acc_src={acc_src:.2f}  acc_tgt={acc_tgt:.2f}")

In [ ]:
# --- Retrain Agent on Source with Reward Offset ---
print("Retraining with Adapted Rewards...")
adapted_agent = Agent(4, 2)
env = make_env(gravity=9.8)
adapted_rewards = []

for ep in range(150):
    reset_out = env.reset()
    state = reset_out[0] if isinstance(reset_out, tuple) else reset_out
    total_r = 0
    done = False
    while not done:
        action = adapted_agent.act(state)
        next_out = env.step(action)
        next_state = next_out[0]
        reward = next_out[1]
        terminated = next_out[2]
        truncated = next_out[3] if len(next_out) > 3 else False
        done = terminated or truncated

        # --- CALCULATE REWARD OFFSET ---
        with torch.no_grad():
            s_t = torch.FloatTensor(state).unsqueeze(0)
            a_t = torch.FloatTensor([action]).unsqueeze(1)
            ns_t = torch.FloatTensor(next_state).unsqueeze(0)

            prob_target = discriminator(s_t, a_t, ns_t).item()
            prob_target = np.clip(prob_target, 0.01, 0.99)
            delta_r = np.log(prob_target / (1 - prob_target))
            modified_reward = reward + 0.1 * delta_r  # scale down to keep stable

        adapted_agent.remember(state, action, modified_reward, next_state, done)
        adapted_agent.train_step()
        state = next_state
        total_r += reward  # logging original reward for clarity

    if ep % 10 == 0:
        adapted_agent.update_target()
    adapted_rewards.append(total_r)

# Evaluate adapted agent on target domain
score_adapted = evaluate(adapted_agent, gravity=25.0)
print(f"Adapted Agent Score: {score_adapted:.2f}")

# Final Comparison Plot
plt.figure(figsize=(10, 5))
plt.bar(
    ["Baseline (No Transfer)", "Domain Randomized", "Dynamics Adapted (Reward Shaping)"],
    [score_baseline, score_random, score_adapted],
    color=['red', 'green', 'blue']
)
plt.title("Comparison of Transfer Learning Methods on High Gravity Domain")
plt.ylabel("Performance (Average Return)")
plt.show()

# Plot adapted training curve to inspect learning
plt.figure(figsize=(10,4))
plot_rewards(adapted_rewards, label='Adapted (reward-shaping)')
plt.title('Adapted Agent Training Returns (smoothed)')
plt.show()

### Notes & Caveats 💡

- This notebook is pedagogical. In practice, use better function approximation, stable training schedules, and more data when training the discriminator.
- The reward offset uses the discriminator's log-odds; clipping and scaling are important for stability.
- Domain randomization is a simple but powerful baseline for many sim-to-real problems.

If you want, I can run a short smoke test executing key cells or tune hyperparameters to reduce training time for interactive use. ✅